# 使用 Conv-SNN 进行 STAG 单帧压力图分类

本 Notebook 是训练和验证的唯一入口。每个样本只包含一张压力图；SNN 的时间维度由模型内部的泊松频率编码生成。

## 重要验证说明

按照当前实验方案，官方 `test` 划分会在每个 epoch 中作为验证集，并参与选择 `best_model.pt`。这会引入模型选择偏差，因此最佳验证分数**不能**视为无偏的最终测试结果。

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score
from spikingjelly.activation_based import functional
from torch import nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

from data import create_single_frame_datasets
from model import SingleFrameConvSNN, count_trainable_parameters

# 进行完整训练前，将此值改为 "full"。
RUN_MODE = "smoke"
RUN_MODE = os.environ.get("SNN_RUN_MODE", RUN_MODE).strip().lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE 必须为 'smoke' 或 'full'。")

SEED = 42
LEARNING_RATE = 1e-3
GAUSSIAN_NOISE_STD = 0.015
DROPOUT = 0.2
TAU = 2.0
PROGRESS_UPDATE_INTERVAL = 10
CUDA_DEVICE_NAME: str | None = None
GPU_MEMORY_GIB: float | None = None

if RUN_MODE == "smoke":
    EPOCHS = 1
    BATCH_SIZE = 4
    TIME_STEPS = 2
    SMOKE_TRAIN_SAMPLES = 4
    SMOKE_VALIDATION_SAMPLES = 4
    DEVICE = torch.device("cpu")
    NUM_WORKERS = 0
    PREFETCH_FACTOR = None
    PIN_MEMORY = False
    AMP_DTYPE = None
    VALIDATION_BATCH_SIZE = BATCH_SIZE
    EXPERIMENT_NAME = "single_frame_smoke"
else:
    EPOCHS = 200
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE.type == "cuda":
        properties = torch.cuda.get_device_properties(0)
        CUDA_DEVICE_NAME = properties.name
        GPU_MEMORY_GIB = properties.total_memory / (1024 ** 3)
        default_batch_size = 512 if GPU_MEMORY_GIB >= 40 else 256
    else:
        default_batch_size = 64
    BATCH_SIZE = int(
        os.environ.get("SNN_BATCH_SIZE", str(default_batch_size))
    )
    VALIDATION_BATCH_SIZE = int(
        os.environ.get(
            "SNN_VALIDATION_BATCH_SIZE",
            str(min(BATCH_SIZE * 2, 1024)),
        )
    )
    TIME_STEPS = int(os.environ.get("SNN_TIME_STEPS", "16"))
    default_workers = 0 if os.name == "nt" else min(
        8, max(1, (os.cpu_count() or 2) - 2)
    )
    NUM_WORKERS = int(
        os.environ.get("SNN_NUM_WORKERS", str(default_workers))
    )
    PREFETCH_FACTOR = 4 if NUM_WORKERS > 0 else None
    PIN_MEMORY = DEVICE.type == "cuda"
    if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
        AMP_DTYPE = torch.bfloat16
    elif DEVICE.type == "cuda":
        AMP_DTYPE = torch.float16
    else:
        AMP_DTYPE = None
    EXPERIMENT_NAME = "single_frame_full"

FUSED_ADAM = DEVICE.type == "cuda"
NON_BLOCKING = PIN_MEMORY
if DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

DATA_PATH = Path(
    os.environ.get(
        "STAG_DATA_PATH", "../stag_data/classification_lite.zip"
    )
).resolve()
OUTPUT_ROOT = Path(os.environ.get("SNN_OUTPUT_ROOT", "outputs")).resolve()
OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_WARNING = (
    "官方测试集在每个 epoch 中被用作验证集，并参与最佳模型选择；"
    "这会引入模型选择偏差，因此最佳验证分数不是无偏的最终测试估计。"
)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

set_seed(SEED)
sns.set_theme(style="whitegrid")
print(f"运行模式：{RUN_MODE}")
print(f"计算设备：{DEVICE}")
if CUDA_DEVICE_NAME is not None:
    print(f"GPU：{CUDA_DEVICE_NAME}（{GPU_MEMORY_GIB:.1f} GiB）")
print(f"批次大小：{BATCH_SIZE}")
print(f"验证批次大小：{VALIDATION_BATCH_SIZE}")
print(f"数据加载进程：{NUM_WORKERS}")
print(f"混合精度：{AMP_DTYPE}")
print(f"数据路径：{DATA_PATH}")
print(f"输出路径：{OUTPUT_DIR}")
print(f"警告：{VALIDATION_WARNING}")

## 1. 读取官方单帧数据划分

In [ ]:
metadata, splits, full_train_dataset, full_validation_dataset = (
    create_single_frame_datasets(DATA_PATH)
)

assert metadata.num_frames == 135_187
assert int(metadata.sensor_mask.sum()) == 548
assert splits.num_classes == 26
assert len(full_train_dataset) == 35_178
assert len(full_validation_dataset) == 15_522
assert "empty_hand" not in splits.class_names

split_summary = pd.DataFrame(
    {
        "数据划分": ["训练集", "验证集（官方测试集）"],
        "样本数": [len(full_train_dataset), len(full_validation_dataset)],
        "每类样本数": [1_353, 597],
        "类别数": [splits.num_classes, splits.num_classes],
    }
)
display(split_summary)

class_mapping = [
    {
        "label": label,
        "original_object_id": original_id,
        "name": splits.class_names[label],
    }
    for label, original_id in enumerate(splits.original_object_ids)
]
(OUTPUT_DIR / "class_mapping.json").write_text(
    json.dumps(class_mapping, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
cache_mib = (
    full_train_dataset.cache_size_bytes
    + full_validation_dataset.cache_size_bytes
) / (1024 ** 2)
print(f"类别名称：{splits.class_names}")
print(f"归一化图像缓存：{cache_mib:.1f} MiB")

## 2. 查看单帧压力图

In [ ]:
sample_indices = np.linspace(
    0, len(full_train_dataset) - 1, num=6, dtype=int
)
fig, axes = plt.subplots(2, 3, figsize=(10, 7), constrained_layout=True)
for axis, dataset_index in zip(axes.ravel(), sample_indices):
    image, label = full_train_dataset[int(dataset_index)]
    view = axis.imshow(image[0], cmap="magma", vmin=0.0, vmax=1.0)
    axis.set_title(splits.class_names[int(label)])
    axis.set_axis_off()
fig.colorbar(view, ax=axes.ravel().tolist(), shrink=0.75, label="Normalized pressure")
fig.suptitle("Example STAG Single-Frame Pressure Maps", fontsize=14)
display(fig)
plt.close(fig)

## 3. 构建数据加载器

完整模式在 Linux 上使用多进程预取、固定内存和持久 worker，以持续向 GPU 供给数据。批次大小可通过 `SNN_BATCH_SIZE` 环境变量覆盖。

In [ ]:
if RUN_MODE == "smoke":
    train_indices = np.linspace(
        0,
        len(full_train_dataset) - 1,
        num=SMOKE_TRAIN_SAMPLES,
        dtype=int,
    ).tolist()
    validation_indices = np.linspace(
        0,
        len(full_validation_dataset) - 1,
        num=SMOKE_VALIDATION_SAMPLES,
        dtype=int,
    ).tolist()
    train_dataset = Subset(full_train_dataset, train_indices)
    validation_dataset = Subset(
        full_validation_dataset, validation_indices
    )
else:
    train_dataset = full_train_dataset
    validation_dataset = full_validation_dataset

loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "persistent_workers": NUM_WORKERS > 0,
    "worker_init_fn": seed_worker,
}
if PREFETCH_FACTOR is not None:
    loader_options["prefetch_factor"] = PREFETCH_FACTOR

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    **loader_options,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    **loader_options,
)

images, labels = next(iter(train_loader))
assert images.ndim == 4 and tuple(images.shape[1:]) == (1, 32, 32)
assert labels.ndim == 1
assert torch.isfinite(images).all()
assert images.min() >= 0 and images.max() <= 1
invalid_mask = torch.from_numpy(~metadata.sensor_mask).unsqueeze(0)
assert torch.count_nonzero(images[:, :, invalid_mask[0]]) == 0
print(f"本次训练样本数：{len(train_dataset)}")
print(f"本次验证样本数：{len(validation_dataset)}")
print(f"批次形状：{tuple(images.shape)}")
print(
    f"DataLoader：workers={NUM_WORKERS}，pin_memory={PIN_MEMORY}，"
    f"prefetch_factor={PREFETCH_FACTOR}"
)

## 4. 创建单帧 Conv-SNN

In [ ]:
model = SingleFrameConvSNN(
    num_classes=splits.num_classes,
    time_steps=TIME_STEPS,
    tau=TAU,
    dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    fused=FUSED_ADAM,
)
grad_scaler = torch.amp.GradScaler(
    "cuda", enabled=AMP_DTYPE == torch.float16
)
criterion = nn.CrossEntropyLoss()
sensor_mask_tensor = (
    torch.from_numpy(metadata.sensor_mask.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(DEVICE)
)
performance_config = {
    "device": str(DEVICE),
    "cuda_device_name": CUDA_DEVICE_NAME,
    "gpu_memory_gib": GPU_MEMORY_GIB,
    "batch_size": BATCH_SIZE,
    "validation_batch_size": VALIDATION_BATCH_SIZE,
    "time_steps": TIME_STEPS,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "pin_memory": PIN_MEMORY,
    "persistent_workers": NUM_WORKERS > 0,
    "amp_dtype": str(AMP_DTYPE) if AMP_DTYPE is not None else None,
    "tf32": DEVICE.type == "cuda",
    "cudnn_benchmark": DEVICE.type == "cuda",
    "fused_adam": FUSED_ADAM,
    "normalized_cache_mib": cache_mib,
}

model.eval()
set_seed(SEED)
try:
    with torch.no_grad():
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=AMP_DTYPE is not None,
        ):
            shape_check = model(
                images.to(DEVICE, non_blocking=NON_BLOCKING)
            )
finally:
    functional.reset_net(model)
assert tuple(shape_check.shape) == (len(labels), splits.num_classes)
assert torch.isfinite(shape_check).all()
print(model)
print(f"可训练参数量：{count_trainable_parameters(model):,}")
print(f"输出形状：{tuple(shape_check.shape)}")
print(f"Fused Adam：{FUSED_ADAM}")

## 5. 定义训练与验证循环

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    num_classes: int,
    description: str,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, object]:
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = torch.zeros((), device=device, dtype=torch.float32)
    top1_correct = torch.zeros((), device=device, dtype=torch.int64)
    top3_correct = torch.zeros((), device=device, dtype=torch.int64)
    sample_count = 0
    all_targets: list[torch.Tensor] = []
    all_predictions: list[torch.Tensor] = []
    epoch_start = time.perf_counter()

    progress_bar = tqdm(
        loader,
        desc=description,
        unit="批次",
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )
    for batch_index, (batch_images, batch_labels) in enumerate(
        progress_bar, start=1
    ):
        batch_images = batch_images.to(
            device, non_blocking=NON_BLOCKING
        )
        batch_labels = batch_labels.to(
            device, non_blocking=NON_BLOCKING
        )

        if is_training:
            noise = torch.randn_like(batch_images) * GAUSSIAN_NOISE_STD
            batch_images = (batch_images + noise).clamp(0.0, 1.0)
            batch_images = batch_images * sensor_mask_tensor
            optimizer.zero_grad(set_to_none=True)

        try:
            with torch.set_grad_enabled(is_training):
                with torch.autocast(
                    device_type=device.type,
                    dtype=AMP_DTYPE,
                    enabled=AMP_DTYPE is not None,
                ):
                    logits = model(batch_images)
                    loss = criterion(logits, batch_labels)
                if is_training:
                    if grad_scaler.is_enabled():
                        grad_scaler.scale(loss).backward()
                        grad_scaler.step(optimizer)
                        grad_scaler.update()
                    else:
                        loss.backward()
                        optimizer.step()
        finally:
            functional.reset_net(model)

        batch_size = batch_labels.numel()
        sample_count += batch_size
        total_loss += loss.detach().float() * batch_size
        predictions = logits.detach().argmax(dim=1)
        top1_correct += (predictions == batch_labels).sum()
        top3 = logits.detach().topk(k=3, dim=1).indices
        top3_correct += (
            (top3 == batch_labels.unsqueeze(1)).any(dim=1).sum()
        )
        all_targets.append(batch_labels.detach())
        all_predictions.append(predictions)

        if (
            batch_index % PROGRESS_UPDATE_INTERVAL == 0
            or batch_index == len(loader)
        ):
            progress_bar.set_postfix(
                loss=f"{loss.detach().float().item():.4f}"
            )

    metric_totals = torch.stack(
        (
            total_loss,
            top1_correct.float(),
            top3_correct.float(),
        )
    ).cpu().numpy()
    targets = torch.cat(all_targets).cpu().numpy()
    predictions = torch.cat(all_predictions).cpu().numpy()
    elapsed_seconds = time.perf_counter() - epoch_start
    matrix = confusion_matrix(
        targets, predictions, labels=np.arange(num_classes)
    )
    macro_f1 = f1_score(
        targets,
        predictions,
        labels=np.arange(num_classes),
        average="macro",
        zero_division=0,
    )
    return {
        "loss": float(metric_totals[0]) / sample_count,
        "top1": float(metric_totals[1]) / sample_count,
        "top3": float(metric_totals[2]) / sample_count,
        "macro_f1": float(macro_f1),
        "sample_count": sample_count,
        "seconds": elapsed_seconds,
        "samples_per_second": sample_count / elapsed_seconds,
        "confusion_matrix": matrix,
    }


def scalar_metrics(metrics: dict[str, object]) -> dict[str, float | int]:
    return {
        "loss": float(metrics["loss"]),
        "top1": float(metrics["top1"]),
        "top3": float(metrics["top3"]),
        "macro_f1": float(metrics["macro_f1"]),
        "sample_count": int(metrics["sample_count"]),
        "seconds": float(metrics["seconds"]),
        "samples_per_second": float(metrics["samples_per_second"]),
    }


def checkpoint_payload(
    epoch: int,
    metrics: dict[str, object],
) -> dict[str, object]:
    return {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": model.get_config(),
        "performance_config": performance_config,
        "class_names": list(splits.class_names),
        "validation_metrics": scalar_metrics(metrics),
        "validation_uses_official_test": True,
        "validation_warning": VALIDATION_WARNING,
    }


def is_better_validation(
    metrics: dict[str, object],
    best_top1: float,
    best_loss: float,
) -> bool:
    top1 = float(metrics["top1"])
    loss = float(metrics["loss"])
    return top1 > best_top1 or (
        np.isclose(top1, best_top1, rtol=0.0, atol=1e-12)
        and loss < best_loss
    )

## 6. 执行训练与验证

冒烟模式仅使用少量真实数据运行一个 epoch；完整模式按配置运行 200 个 epoch。训练与验证均使用 `tqdm` 进度条，并控制 loss 刷新频率以减少 GPU 同步。

In [ ]:
set_seed(SEED)
history: list[dict[str, float | int]] = []
best_epoch = -1
best_top1 = float("-inf")
best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        DEVICE,
        splits.num_classes,
        description=f"训练 {epoch}/{EPOCHS}",
        optimizer=optimizer,
    )
    validation_metrics = run_epoch(
        model,
        validation_loader,
        criterion,
        DEVICE,
        splits.num_classes,
        description=f"验证 {epoch}/{EPOCHS}",
    )

    row: dict[str, float | int] = {"epoch": epoch}
    for prefix, metrics in (
        ("train", train_metrics),
        ("validation", validation_metrics),
    ):
        for name, value in scalar_metrics(metrics).items():
            row[f"{prefix}_{name}"] = value
    history.append(row)

    if is_better_validation(validation_metrics, best_top1, best_loss):
        best_epoch = epoch
        best_top1 = float(validation_metrics["top1"])
        best_loss = float(validation_metrics["loss"])
        torch.save(
            checkpoint_payload(epoch, validation_metrics),
            OUTPUT_DIR / "best_model.pt",
        )

    print(
        f"轮次 {epoch:03d}/{EPOCHS:03d} | "
        f"训练损失={train_metrics['loss']:.4f}，"
        f"Top-1={train_metrics['top1']:.3f} | "
        f"验证损失={validation_metrics['loss']:.4f}，"
        f"Top-1={validation_metrics['top1']:.3f}，"
        f"Top-3={validation_metrics['top3']:.3f} | "
        f"训练吞吐={train_metrics['samples_per_second']:.1f} 样本/秒，"
        f"验证吞吐={validation_metrics['samples_per_second']:.1f} 样本/秒"
    )

torch.save(
    checkpoint_payload(EPOCHS, validation_metrics),
    OUTPUT_DIR / "last_model.pt",
)
history_frame = pd.DataFrame(history)
history_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)
assert (OUTPUT_DIR / "best_model.pt").is_file()
assert (OUTPUT_DIR / "last_model.pt").is_file()
display(history_frame)

## 7. 加载最佳模型并可视化结果

In [ ]:
best_checkpoint = torch.load(
    OUTPUT_DIR / "best_model.pt",
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
set_seed(SEED)
best_validation_metrics = run_epoch(
    model,
    validation_loader,
    criterion,
    DEVICE,
    splits.num_classes,
    description="复核最佳模型",
)
print(f"选中的最佳轮次：{best_checkpoint['epoch']}")
print(f"重新计算的验证指标：{scalar_metrics(best_validation_metrics)}")
print(f"警告：{VALIDATION_WARNING}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
plots = (
    ("loss", "Cross-Entropy Loss", "Loss"),
    ("top1", "Top-1 Accuracy", "Accuracy"),
    ("top3", "Top-3 Accuracy", "Accuracy"),
    ("macro_f1", "Macro F1", "Score"),
)
for axis, (column, title, ylabel) in zip(axes.ravel(), plots):
    axis.plot(
        history_frame["epoch"],
        history_frame[f"train_{column}"],
        marker="o",
        label="Train",
    )
    axis.plot(
        history_frame["epoch"],
        history_frame[f"validation_{column}"],
        marker="o",
        label="Validation (Official Test)",
    )
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_ylabel(ylabel)
    axis.legend()
fig.suptitle("Single-Frame Conv-SNN Training History", fontsize=15)
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
matrix = np.asarray(best_validation_metrics["confusion_matrix"])
fig, axis = plt.subplots(figsize=(14, 12), constrained_layout=True)
sns.heatmap(
    matrix,
    cmap="Blues",
    xticklabels=splits.class_names,
    yticklabels=splits.class_names,
    cbar_kws={"label": "Frame count"},
    ax=axis,
)
axis.set_title("Validation Confusion Matrix (Official Test Split)")
axis.set_xlabel("Predicted object")
axis.set_ylabel("True object")
axis.tick_params(axis="x", rotation=90)
axis.tick_params(axis="y", rotation=0)
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
row_totals = matrix.sum(axis=1)
class_accuracy = np.divide(
    np.diag(matrix),
    row_totals,
    out=np.zeros(splits.num_classes, dtype=np.float64),
    where=row_totals > 0,
)
fig, axis = plt.subplots(figsize=(14, 6), constrained_layout=True)
axis.bar(splits.class_names, class_accuracy, color="#4472C4")
axis.set_title("Per-Class Validation Accuracy (Official Test Split)")
axis.set_xlabel("Object class")
axis.set_ylabel("Accuracy")
axis.set_ylim(0.0, 1.0)
axis.tick_params(axis="x", rotation=90)
fig.savefig(OUTPUT_DIR / "class_accuracy.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

## 8. 保存实验摘要

In [ ]:
summary = {
    "run_mode": RUN_MODE,
    "num_frames": metadata.num_frames,
    "sensor_count": int(metadata.sensor_mask.sum()),
    "num_classes": splits.num_classes,
    "class_names": list(splits.class_names),
    "full_train_samples": len(full_train_dataset),
    "full_validation_samples": len(full_validation_dataset),
    "used_train_samples": len(train_dataset),
    "used_validation_samples": len(validation_dataset),
    "epochs": EPOCHS,
    "best_epoch": int(best_checkpoint["epoch"]),
    "selected_validation_metrics": best_checkpoint["validation_metrics"],
    "reevaluated_validation_metrics": scalar_metrics(
        best_validation_metrics
    ),
    "model_config": model.get_config(),
    "performance_config": performance_config,
    "validation_uses_official_test": True,
    "validation_warning": VALIDATION_WARNING,
    "output_files": [
        "best_model.pt",
        "last_model.pt",
        "history.csv",
        "summary.json",
        "class_mapping.json",
        "training_curves.png",
        "confusion_matrix.png",
        "class_accuracy.png",
    ],
}
(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

expected_outputs = [OUTPUT_DIR / name for name in summary["output_files"]]
missing_outputs = [path.name for path in expected_outputs if not path.is_file()]
if missing_outputs:
    raise AssertionError(f"缺少输出文件：{missing_outputs}")

display(pd.DataFrame([summary["reevaluated_validation_metrics"]]))
print(f"实验结果已保存至：{OUTPUT_DIR}")
print(f"警告：{VALIDATION_WARNING}")

## 结果解释

这是一个使用 Nature 论文官方单帧划分和平衡标记、并移除 `empty_hand` 的轻量 Conv-SNN 基线。它不是对论文改进 ResNet 或论文 27 类结果的严格复现。由于官方测试集参与选择最佳模型，报告结果时应将其表述为存在模型选择偏差的验证性能。